In [ ]:
%reload_ext autoreload
%autoreload 2

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

import os
import logging
import yaml
import pickle
import ipynbname

import pandas as pd
import numpy as np

import ferdinand.utils as utl   
import ferdinand.sqlite_utils as sqlutl
import ferdinand.clustering_utils as clutl

from pathlib import Path
from types import SimpleNamespace

from sklearn.neighbors import kneighbors_graph
import igraph as ig
import leidenalg as la

import matplotlib.pyplot as plt
import warnings

# Purpose and Documentation

## Purpose

This notebook uses <b>_Leiden_ clustering</b> to automatically group preprocessed radiographs by visual similarity, then enables manual validation and curation to exclude non-whole-body X-rays and low-quality images. The curated and validated datasets produced here serve as input for downstream ML pipeline stages.

## Location 

_&lt;project_dir&gt;/notebooks/310_curate_dataset.ipynb_

## Prerequisites

- A valid runtime configuration file (_../config/config.yml_) with correctly defined _centers_, _run_name_, _parameters_, and _directories_
- Successful execution of the template setup cells from _NNN_notebook_template.ipynb_
- Existing _../data/_ and _../reports/_ directories 
- Existing SQLite database _../data/&lt;run_name&gt;/&lt;all_data_dir&gt;/&lt;sqlite_db&gt;.db_
- Image files have already been preprocessed and are present in _../data/&lt;run_name&gt;/&lt;preprocessed_dir&gt;_

## Workflow

- Extract image features (CNN-based, using SWIN or CLIP models)
- Build k-nearest-neighbor graphs per imaging center
- Apply Leiden community detection to identify clusters of visually similar images
- Visualize clusters using PCA and UMAP
- Generate interactive HTML galleries per cluster for manual review
- Mark clusters/images for exclusion based on QC criteria
- Update SQLite database with cluster assignments and curation decisions


# Definition / Settings

This section initializes the notebook environment: 
- loads the runtime configuration from _../config/config.yml_
- creates the required directory structure
- sets up logging and the SQLite connection

**Do not proceed past this section** </br>
Missing paths and an uninitialized database connection will cause subsequent cells to crash.

## General settings

In [ ]:
# load the configuration file
config_path = Path('../config/config.yml')
with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

print(pd.DataFrame(config['settings'].items(), columns=['configuration parameter', 'set to']).to_markdown(tablefmt="pretty", index=False))

## Set up runtime paths

In [ ]:
# create directories as per configuration
dirs = SimpleNamespace(**config['data_directories'])
for key, value in vars(dirs).items():
    dirs.__dict__[key] = os.path.join(config['project_root'], config['data_dir'], config['run_name'], value)
    # os.makedirs(dirs.__dict__[key], exist_ok=True)
    if os.path.exists(dirs.__dict__[key]):
        print(f" Directory created: {dirs.__dict__[key]}")
    else:
        print(f" ERROR: Directory does not exist: {dirs.__dict__[key]}")

## Store runtime configuration when necessary

In [ ]:
# copy the current '../config/config.yml' to '../reports/'
dest_filename = f"config_{config['run_name']}.yml"
dest_path = os.path.join(config['project_root'], config['reports_dir'], config['run_name'], dest_filename)
if not os.path.exists(dest_path): 
    commmand_string = f"cp {config_path} {dest_path}"
    os.system(commmand_string)

    if os.path.exists(dest_path): 
        print(f" Current runtime configuration \'{config_path}' saved to \'{dest_path}\'")
    else:
        print(f" ERROR: could not save configuration file to \'{dest_path}\'")

## Set up logger

In [ ]:
# setup logging
notebook_name = ipynbname.name()
notebook_idx = notebook_name.split('_')[0]

log_path = os.path.join(config['project_root'], config['reports_dir'], config['run_name'], 'logs')
os.makedirs(log_path, exist_ok=True)

log_file = os.path.join(log_path, f"{notebook_name}.log")
print(f"Log file: {log_file}")

try:
    logger = utl.setup_logger(log_file, log_level=logging.INFO)
    print("Logger successfully set up.")
except Exception as e:
    print(f"Error setting up logger: {e}")

## Set up SQLite connection

In [ ]:
# expected SQLlite database file
sqlite_file = os.path.join(dirs.all_data_dir, f"{config['sqlite_db']}.db")

# if already exists, connect to it
if os.path.exists(sqlite_file):
    conn = sqlutl.connect_db(sqlite_file)
    if conn is not None:
        print(f"connected to database {sqlite_file}\n")
else: 
    print(f" ERROR: No SQLite database found!")

# Build _Leiden_ clusters per center

This section extracts image features (CNN embeddings), builds k-nearest-neighbor graphs, applies Leiden community detection, and visualizes the resulting clusters.

## Workflow

- Extract features (optional): Use SWIN or CLIP model to compute embeddings (1024-dim or 512-dim vectors) for all preprocessed images. Save as _.npy_ files.
- Load features: Reuse pre-computed features from previous runs if available (much faster).
- Build KNN graph: For each center, construct a k-nearest-neighbor connectivity matrix, then convert to igraph object.
- Apply _Leiden_: Search over resolution parameters to find optimal cluster count and modularity. Save cluster labels.
- Visualize: Create 2D projections (PCA, UMAP) colored by cluster ID to inspect separation quality.
- Generate galleries: Create interactive HTML files showing thumbnail images per cluster for manual review.

## Key parameters
- n_neighbors: Number of nearest neighbors to connect in KNN graph (typically 3–300 depending on dataset size).
- resolution: _Leiden_ algorithm parameter controlling cluster granularity (0.1–2.0). Lower = fewer clusters.
- seed: Random seed for reproducibility (set to 42).

**Feature extraction is computationally expensive (~1–10 hours for 10k+ images). Always save and reuse features rather than re-computing.**

In [ ]:
# create directory for cluster analysis results
clu_analysis_dir = os.path.join(config['project_root'], config['reports_dir'], config['run_name'], "cluster_analysis")
os.makedirs(clu_analysis_dir, exist_ok=True)
if os.path.exists(clu_analysis_dir):
    print(f" Directory created: {clu_analysis_dir}")
else:
    print(f" ERROR: Directory does not exist: {clu_analysis_dir}")

In [ ]:
# load centers from config file
my_centers = config['centers']
print(f"Centers to be processed: {my_centers}")

## Extract features

**Feature extraction is slow**: Extracting features for 10k+ images can take hours. Pre-compile once and reuse.

In [ ]:
# initialize a dictionary to store clustering results
clu_results = {
    'paths': {},          # file paths
    'features': {},       # feature matrices
    'partitions': {},     # cluster assignments
    'resolutions': {},    # best resolution parameters
    'n_neighbors': {},    # best n_neighbors parameters
    'modularities': {},   # modularity scores
    'n_clusters': {},     # number of clusters
    'history': {}         # optimization history (optional)
}

In [ ]:
# extract features for each center
for center in my_centers:

    # select images 
    image_files = utl.select_jpegs_by_column(conn=conn, 
                                             db_table='impc_data',
                                             column='center',
                                             value=center, 
                                             source_path=dirs.preprocessed_dir)

    if len(image_files) > 0:
        # extract features using microsoft/swin-tiny-patch4-window7-224
        clu_results['paths'][center], clu_results['features'][center] = clutl.extract_features(image_files=image_files)

        print(f"Extracted features for center: {center}, number of samples: {clu_results['features'][center].shape[0]}, feature dimension: {clu_results['features'][center].shape[1]}")
    else:
        print(f"No files found for center: {center}")

print(f"Features extracted for centers: {list(clu_results['features'].keys())}")
print(f"Feature shapes for centers: { {center: clu_results['features'][center].shape for center in clu_results['features'].keys()} }")

In [ ]:
# save the results to a pickle file for later use
with open(os.path.join(clu_analysis_dir, 'cluster_analysis_results.pkl'), 'wb') as f:
    pickle.dump(clu_results, f)

## Load extracted features

In [ ]:
# load the results from the pickle file
with open(os.path.join(clu_analysis_dir, 'cluster_analysis_results.pkl'), 'rb') as f:
    clu_results = pickle.load(f)

for center in clu_results['features'].keys():
    print(f"Feature shape for centers {center}: {clu_results['features'][center].shape}")

## Build graph and apply _Leiden_

In [ ]:
# perform clustering optimization for each center
for center in clu_results['features'].keys():
    # perform clustering optimization
    result = clutl.find_best_parameters_bayesian(
        features=clu_results['features'][center],
        n_neighbors_range=(3, 19), # 30
        resolution_range=(0.1, 2.0),
        n_calls=10,          # More calls = better (but slower)
        n_random_starts=2,   # Random exploration at the start
        repeats=3,           # Stability through repetitions
        seed=42,
        verbose=True
    )   

    # Display results
    print("\n" + "="*60)
    print(f"CLUSTERING RESULTS FOR CENTER: {center}")
    print("\n" + "="*60)
    print(" BEST PARAMETERS:")
    print("="*60)
    print(f" n_neighbors:  {result['best_n_neighbors']}")
    print(f" resolution:   {result['best_resolution']:.4f}")
    print(f" modularity:   {result['best_modularity']:.4f}")
    print("="*60)

    # store picklable cluster labels instead of the non-picklable leiden partition object
    best_partition = result['best_partition']
    if hasattr(best_partition, 'membership'):
        clu_results['partitions'][center] = list(best_partition.membership)
    else:
        clu_results['partitions'][center] = list(best_partition)

    clu_results['resolutions'][center] = result['best_resolution']
    clu_results['n_neighbors'][center] = result['best_n_neighbors']
    clu_results['modularities'][center] = result['best_modularity']
    clu_results['history'][center] = result['history']

    center_stripped = center.replace(" ", "_")
    output_dir = os.path.join(clu_analysis_dir, center_stripped)
    os.makedirs(output_dir, exist_ok=True)

    # plot optimization results
    clutl.plot_optimization_results(result, output_dir=output_dir)
    
    # sensitivity analysis
    sensitivity = clutl.analyze_sensitivity(clu_results['features'][center], result, output_dir=output_dir)

In [ ]:
# check if the results are picklable
for key, value in clu_results.items():
    try:
        pickle.dumps(value)
        print(f"✓ {key} is picklable")
    except Exception as e:
        print(f"✗ {key} failed: {e}")

In [ ]:
# save final results
with open(os.path.join(clu_analysis_dir, 'cluster_analysis_results.pkl'), 'wb') as f:
    pickle.dump(clu_results, f)

## Load existing clusters

In [ ]:
# load the results from the pickle file
with open(os.path.join(clu_analysis_dir, 'cluster_analysis_results.pkl'), 'rb') as f:
    clu_results = pickle.load(f)

for center in clu_results['features'].keys():
    print(f"Feature shape for center {center}: {clu_results['features'][center].shape}")
    clu_results['n_clusters'][center] = len(set(clu_results['partitions'][center]))
    print(f"Number of clusters for center {center}: {clu_results['n_clusters'][center]}\n")

## PCA

In [ ]:
# visualize the clusters using PCA
for center in clu_results['features'].keys():

    labels = clu_results['partitions'][center]
    features = clu_results['features'][center]

    output_file = os.path.join(clu_analysis_dir, str(center).replace(" ", "_").replace("/", "_"), "leiden_clustering_PCA.png")

    print(f"Computing and plotting PCA embedding for {center}")
    clutl.plot_embedding(labels=labels, features=features, method="pca", annotate_clusters=False, output_file=None)

## UMAP

In [ ]:
warnings.filterwarnings("ignore", category=UserWarning, module="umap")

# visualize the clusters using UMAP
for center in clu_results['features'].keys():

    labels = clu_results['partitions'][center]
    features = clu_results['features'][center]

    output_file = os.path.join(clu_analysis_dir, str(center).replace(" ", "_").replace("/", "_"), "leiden_clustering_UMAP.png")

    print(f"Plotting UMAP embedding for {center}")
    clutl.plot_embedding(labels=labels, features=features, method="umap", annotate_clusters=False, output_file=None)

# Create a thumbnail gallery for each cluster

**Requirement**: Preprocessed image thumbnails must be generated beforehand.

In [ ]:
# create directory for cluster gallery
clu_gallery_dir = os.path.join(config['project_root'], config['reports_dir'], config['run_name'], "cluster_gallery")
os.makedirs(clu_gallery_dir, exist_ok=True)
if os.path.exists(clu_gallery_dir):
    print(f" Directory created: {clu_gallery_dir}")
else:
    print(f" ERROR: Directory does not exist: {clu_gallery_dir}")

In [ ]:
# generate HTML galleries for each cluster
rows = sqlutl.select_rows(conn, 'impc_data')
rows_indexed = rows.set_index('omero_id')

# build combined dataframe from clu_results
frames = []
for c in clu_results['paths'].keys():
    frames.append(pd.DataFrame({
        'image_path': clu_results['paths'][c],
        'cluster': clu_results['partitions'][c],
        'center': c
    }))
df = pd.concat(frames, ignore_index=True)

df['image_path'] = df['image_path'].astype(str)
df['omero_id'] = (df['image_path']
      .str.split('/')
      .str[-1]
      .str.rsplit('.', n=1)
      .str[0]
      .astype(int))

# create galleries per center/cluster
for center, df_center in df.groupby('center'):
     center_stripped = center.replace(" ", "_")
     n_neighbors = clu_results['n_neighbors'][center]
     resolution = round(clu_results['resolutions'][center], 1)
     for cluster, df_cluster in df_center.groupby('cluster'):
         n_images = len(df_cluster)
         print(f"Creating gallery for center: {center}, cluster: {cluster}, n_neighbors: {n_neighbors}, resolution: {resolution}, number of images: {n_images}")
         omero_ids = df_cluster['omero_id'].unique() 
         html_dir = os.path.join(clu_gallery_dir, center_stripped)
         os.makedirs(html_dir, exist_ok=True)
         utl.create_thumbnail_gallery(rows[rows['omero_id'].isin(omero_ids)], 
                                      f"_cluster_{cluster}_nbrs_{n_neighbors}_res_{resolution}_imgs_{len(df_cluster)}.html",
                                      f"Cluster {cluster}", 
                                      html_dir,
                                      dirs.preprocessed_thumbnails_dir,
                                      dirs.preprocessed_dir)

# Curate dataset based on clustering results

This section manually curates the dataset by reviewing thumbnail galleries and marking clusters/images for exclusion.

## Workflow

- **Review HTML galleries**: Open generated cluster galleries and inspect thumbnail images to identify problematic clusters.
- **Identify exclusion criteria**:
  - Non-whole-body images (e.g. crop, close-up, partial view)
  - Poor contrast, artifacts, or formatting issues
- **Update exclusion dictionaries**
  - _cluster2exclude[center]_: List of clusters to be excluded per center
  - _ids2exclude[center]_: List of specific OMERO IDs to be excluded per center
- **Update SQLite**: For each image, set _leiden_cluster_ column to cluster ID and _keep=no_ if image/cluster is excluded
- **Final dataset**: Rows with _keep=yes_ form the curated dataset

**Important**: Manueal review of cluster galleries is essential, automated clustering alone does not guarantee data quality.

In [ ]:
# add cluster assignments and keep flag to the database
sqlutl.add_column_to_table(conn=conn, db_table='impc_data', column_name='leiden_cluster', column_type='INTEGER', logger=logger)
sqlutl.add_column_to_table(conn=conn, db_table='impc_data', column_name='keep', column_type='TEXT', default_value='yes', logger=logger) # possible values yes/no

In [ ]:
# update the database with cluster assignments
cluster2exclude = {
     'JAX': [17, 18, 19, 20], 
     'MRC Harwell': [19], 
     'UC Davis': [25, 26, 27],
     'WTSI': [19, 20] 
    # add other centers and clusters to exclude as needed
}
ids2exclude = { 'BCM': [596443, 593940, 1246602, 1572127],
                'MRC Harwell': [1082585], 
                'TCP': [482492, 482607], 
                'UC Davis': [1189495, 1189947, 1066105] 
                # add other centers and omero_ids to exclude as needed
               } 

In [ ]:
# update database with cluster assignments and keep flag
for center in clu_results['features'].keys():
    df = pd.DataFrame({
        "image_path": clu_results['paths'][center],
        "cluster": clu_results['partitions'][center]
    })
    
    # extract OMERO IDs once
    df['omero_id'] = df['image_path'].apply(
        lambda path: int(os.path.splitext(os.path.basename(path))[0])
    )
    
    # determine which images to exclude
    exclude_by_cluster = df['cluster'].isin(cluster2exclude[center]) if center in cluster2exclude else pd.Series([False]*len(df))
    exclude_by_id = df['omero_id'].isin(ids2exclude[center]) if center in ids2exclude else pd.Series([False]*len(df))
    df['keep'] = ~(exclude_by_cluster | exclude_by_id)
    
    # update database
    for _, row in df.iterrows():
        sqlutl.update_column_values(conn, 'impc_data', 'leiden_cluster', int(row['cluster']), 'omero_id', int(row['omero_id']))
        if not row['keep']:
            print(f"{row['omero_id']} not kept")
            sqlutl.update_column_values(conn, 'impc_data', 'keep', 'no', 'omero_id', int(row['omero_id']))

# Create thumbnail gallery

Summarizes the curated dataset and creates an interactive HTML gallery of thumbnails that link to the full preprocessed images for quick manual review.

**Requirement**: Preprocessed image thumbnails must be generated beforehand.

In [ ]:
# create an HTML gallery directory
gallery_dir = os.path.join(config['project_root'], config['reports_dir'], config['run_name'], 'thumbnails_gallery')
os.makedirs(gallery_dir, exist_ok=True)

## Curated images

In [ ]:
# get list of rows to keep
rows2keep = sqlutl.select_rows_by_column(conn, 'impc_data', 'keep', 'yes')

In [ ]:
# create final curated dataset gallery
utl.create_thumbnail_gallery(rows2keep, 
                             f"{notebook_idx}_all_curated_images.html",
                             f"Clustering-based Curated Dataset",
                             gallery_dir,
                             dirs.preprocessed_thumbnails_dir,
                             dirs.preprocessed_dir)

In [ ]:
# create HTML galleries for each center in the curated dataset
for center in config['centers']:
    center_df = rows2keep[rows2keep['center'] == center]
    if not center_df.empty:
        center_stripped = center.replace(" ", "_")
        center_gallery_dir = os.path.join(gallery_dir, center_stripped)
        os.makedirs(center_gallery_dir, exist_ok=True)
        utl.create_thumbnail_gallery(
            data=center_df,
            html_file_name=f'{notebook_idx}_all_curated_images_{center_stripped}.html',
            title=f'Clustering-based Curated Images from {center}',
            html_dir=center_gallery_dir,
            thumbnails_dir=dirs.preprocessed_thumbnails_dir,
            source_dir=dirs.preprocessed_dir)

## Excluded images

In [ ]:
# get list of rows to be excluded
rows2exclude = sqlutl.select_rows_by_column(conn, 'impc_data', 'keep', 'no')

In [ ]:
# create gallery for excluded images
utl.create_thumbnail_gallery(rows2exclude.sort_values(by=['center', 'leiden_cluster']),
                             f"{notebook_idx}_excluded_after_curation.html",
                             f"Clustering-based Excluded Images",
                             gallery_dir,
                             dirs.preprocessed_thumbnails_dir,
                             dirs.preprocessed_dir)

In [ ]:
for center in config['centers']:
    center_df = rows2exclude[rows2exclude['center'] == center].sort_values(by=['leiden_cluster'])
    if not center_df.empty:
        center_stripped = center.replace(" ", "_")
        center_gallery_dir = os.path.join(gallery_dir, center_stripped)
        os.makedirs(center_gallery_dir, exist_ok=True)
        utl.create_thumbnail_gallery(
            data=center_df,
            html_file_name=f'{notebook_idx}_excluded_after_curation_{center_stripped}.html',
            title=f'Clustering-based Excluded Images from {center}',
            html_dir=center_gallery_dir,
            thumbnails_dir=dirs.preprocessed_thumbnails_dir,
            source_dir=dirs.preprocessed_dir)